In [ ]:
import pyrealsense2 as rs
import numpy as np
import cv2
from time import time, sleep
from matplotlib import pyplot as plt


def capture_and_save_image():
    print("Using RealSense SDK for L515 camera...")
    
    # Configure the pipeline
    pipeline = rs.pipeline()
    config = rs.config()
    
    # Enable RGB stream
    config.enable_stream(rs.stream.color, 640, 480, rs.format.bgr8, 30)
    
    # Start streaming
    profile = pipeline.start(config)
    
    try:
        # Allow camera to settle
        for i in range(30):
            pipeline.wait_for_frames()
        
        t0 = time()
        
        # Capture a single frame
        frames = pipeline.wait_for_frames()
        color_frame = frames.get_color_frame()
        
        if not color_frame:
            print("Failed to capture color frame")
            return
            
        # Convert to numpy array
        frame = np.asanyarray(color_frame.get_data())
        print(f"Captured frame shape: {frame.shape}")

        # Convert from BGR to RGB for matplotlib
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # Get actual frame dimensions
        frame_height, frame_width = image.shape[:2]
        print(f"Frame dimensions: {frame_width}x{frame_height}")
        
        # Define safer crop coordinates based on actual frame size
        x1, y1 = 400, 0  # Example starting coordinates
        x2, y2 = 1600, 900  # Example ending coordinates
        
        # Clamp coordinates to actual frame size
        x1 = max(0, min(x1, frame_width))
        y1 = max(0, min(y1, frame_height))
        x2 = max(x1 + 1, min(x2, frame_width))  # Ensure x2 > x1
        y2 = max(y1 + 1, min(y2, frame_height))  # Ensure y2 > y1
        
        print(f"Adjusted crop coordinates: ({x1}, {y1}) to ({x2}, {y2})")
        
        # Crop the image
        image = image[y1:y2, x1:x2]
        print(f"Cropped image shape: {image.shape}")
        
        # Only resize if we have a valid cropped image
        if image.size > 0:
            image = cv2.resize(image, (640, 480), interpolation=cv2.INTER_AREA)
            print("Time taken to capture image: {:.4f} seconds".format(time() - t0))
            print(f"Final image shape: {image.shape}")
            
            # Display the captured image
            plt.figure(figsize=(10, 6))
            plt.imshow(image)
            plt.title(f"L515 RGB via RealSense SDK ({frame_width}x{frame_height} → 640x480)")
            plt.axis('off')
            plt.show()
        else:
            print("Error: Cropped image is empty!")

        sleep(0.04)

    finally:
        # Stop streaming
        pipeline.stop()

# Example usage
if __name__ == '__main__':
    capture_and_save_image()

In [ ]:
import cv2
import matplotlib.pyplot as plt
from matplotlib.widgets import RectangleSelector
from time import sleep

def onselect(eclick, erelease):
    """Store the coordinates of the selected rectangle."""
    global x1, y1, x2, y2
    x1, y1 = int(eclick.xdata), int(eclick.ydata)
    x2, y2 = int(erelease.xdata), int(erelease.ydata)
    print(f"Selected region: x1={x1}, y1={y1}, x2={x2}, y2={y2}")
    
    # Show the cropped image
    cropped = frame[y1:y2, x1:x2]
    if cropped.size > 0:  # Check if the crop is valid
        resized = cv2.resize(cropped, (640, 480), interpolation=cv2.INTER_AREA)
        ax2.imshow(resized)
        fig.canvas.draw()
        print(f"Cropped image shape: {resized.shape}")

# Initialize camera with proper backend for L515
cam = cv2.VideoCapture(6, cv2.CAP_V4L2)
if not cam.isOpened():
    raise IOError("Cannot open camera")

# Capture a frame to see the full camera view
_, frame = cam.read()
frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
print(f"Original image shape: {frame.shape}")

# Create figure with two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))
ax1.imshow(frame)
ax1.set_title('Select crop region (click and drag)')
ax2.set_title('Preview of cropped & resized image')
ax2.set_xlim([0, 640])
ax2.set_ylim([480, 0])  # Reversed to match image coordinates

# Set up the rectangle selector
rect_selector = RectangleSelector(
    ax1, onselect, useblit=True,
    button=[1], minspanx=5, minspany=5,
    spancoords='pixels', interactive=True
)

plt.tight_layout()
plt.show()

# Release the camera when done
cam.release()

print("Use these values in your capture_image function:")
print(f"x1, y1 = {90}, {0}  # Top left corner")
print(f"x2, y2 = {600}, {480}  # Bottom right corner")

In [ ]:
import cv2
from time import time, sleep
from matplotlib import pyplot as plt


def capture_and_save_image():
    # Initialize the camera with proper backend for L515
    cam = cv2.VideoCapture(6, cv2.CAP_V4L2)

    # Check if the camera opened successfully
    if not cam.isOpened():
        raise IOError("Cannot open camera")

    for i in range(1):
        t0 = time()
        # Capture a single frame
        _, frame = cam.read()
        

        # Generate a unique filename with the current date and time
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        #image = cv2.resize(image, (640, 480), interpolation=cv2.INTER_AREA)
        # Define your crop coordinates (top left corner and bottom right corner)
        x1, y1 = 400, 000  # Example starting coordinates (top left of the crop rectangle)
        x2, y2 = 1600, 900  # Example ending coordinates (bottom right of the crop rectangle)

        # Crop the image
        image = image[y1:y2, x1:x2]

        image = cv2.resize(image, (640, 480), interpolation=cv2.INTER_AREA)
        print("Time taken to capture image: {:.4f} seconds".format(time() - t0))

        print(image.shape)
        # Save the captured image to the current directory
        plt.imshow(image)
        sleep(0.04)


    # Release the camera
    cam.release()

# Example usage
if __name__ == '__main__':
    capture_and_save_image()